# ShelfPulse AI — YOLO11s/YOLO11m SKU-110K Retail Model Training & Dynamic NCNN Export (v2.3.0)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/shelf-pulse-ai/blob/main/notebooks/train_yolo11_sku110k_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/tahadeab-ai/shelf-pulse-ai)
[![License](https://img.shields.io/badge/License-Apache_2.0-green.svg)](https://opensource.org/licenses/Apache-2.0)

**Project:** ShelfPulse AI (`shelf-pulse-ai` v2.3.0)  
**Target:** Edge-AI Real-Time Retail Shelf Counting (Android arm64-v8a / NCNN Vulkan GPU / ARM NEON FP16)  
**Models Supported:** `YOLO11n` (Fastest, 2.6M params), `YOLO11s` (Recommended Production, 9.4M params), `YOLO11m` (Max Accuracy, 20.1M params)  
**Dataset:** SKU-110K (Dense retail products, 110,000+ dense shelf instances)  
**Resolutions:** Dynamic `320×320` (Live Scan, 2,100 anchors) and `640×640` (Snap & Verify, 8,400 anchors), with optional `800×800` HD option.


## 1. Environment Setup & Toolchain Installation

In [ ]:
!nvidia-smi
!pip install -q --upgrade ultralytics pnnx ncnn onnx huggingface_hub
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## 2. Model Selection & SKU-110K Configuration
Select your model size (`yolo11s.pt` recommended for higher accuracy on dense retail shelves) and input resolution (`640` or `800`).

In [ ]:
# Model Configuration
MODEL_SIZE = "yolo11s.pt" # Options: "yolo11n.pt", "yolo11s.pt", "yolo11m.pt"
IMG_SIZE = 640           # Options: 640 (standard), 800 (ultra-dense)
BATCH_SIZE = 16          # Adjust based on GPU VRAM (16 for T4 16GB at 640px)
EPOCHS = 50              # Fine-tuning epochs

print(f"Configured: Model={MODEL_SIZE}, ImgSize={IMG_SIZE}, Batch={BATCH_SIZE}, Epochs={EPOCHS}")


## 3. Dataset Preparation (SKU-110K)
SKU-110K is the industry benchmark for retail item detection (11,762 images, ~1.7 million bounding boxes). You can train directly on SKU-110K using Ultralytics auto-download dataset configuration.

In [ ]:
import os
import yaml

# Ultralytics provides built-in SKU-110K dataset configuration
# Dataset yaml: SKU-110K.yaml (auto-downloads train/val/test images & annotations)
data_config = "SKU-110K.yaml"
print(f"Using dataset configuration: {data_config}")


## 4. Train High-Accuracy YOLO11 on SKU-110K
Training recipe tuned for tiny dense retail items:
- `mosaic=1.0`: Crucial for learning dense multi-shelf product arrangements.
- `copy_paste=0.3`: Synthetically copies items across shelves to enrich dense layouts.
- `mixup=0.1`: Reduces overfitting on package glare.
- `degrees=0.0`: Products on shelves stand upright (avoids unrealistic orientation drift).
- `fliplr=0.5`: Horizontal flips are valid shelf arrangements.
- `close_mosaic=10`: Disables mosaic in the final 10 epochs for crisp boundary localization.

In [ ]:
from ultralytics import YOLO

# Initialize model from pretrained weights
model = YOLO(MODEL_SIZE)

# Launch training
results = model.train(
    data=data_config,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    mosaic=1.0,
    copy_paste=0.3,
    mixup=0.1,
    degrees=0.0,
    fliplr=0.5,
    close_mosaic=10,
    workers=4,
    device=0,
    project="runs/shelf_pulse",
    name="yolo11_sku110k"
)

# Best weights saved at: runs/shelf_pulse/yolo11_sku110k/weights/best.pt
best_pt = "runs/shelf_pulse/yolo11_sku110k/weights/best.pt"


## 5. Evaluate Validation Metrics (mAP@50 & mAP@50-95)
Compute authentic detection precision, recall, and mAP metrics against the SKU-110K validation set.

In [ ]:
import os
import urllib.request
from ultralytics import YOLO

# Use trained best.pt or pretrained huggingface SKU-110K weights
pt_path = "models/sku110k-yolo11.pt"
os.makedirs("models", exist_ok=True)

if os.path.exists("runs/shelf_pulse/yolo11_sku110k/weights/best.pt"):
    pt_path = "runs/shelf_pulse/yolo11_sku110k/weights/best.pt"
else:
    print("Using authentic HuggingFace pretrained SKU-110K weights...")
    pt_url = "https://huggingface.co/chistopat/sku110k-yolo11-object-detector/resolve/main/weights/sku110k-yolo11-n640.pt"
    urllib.request.urlretrieve(pt_url, pt_path)

eval_model = YOLO(pt_path)
val_results = eval_model.val(data=data_config, imgsz=IMG_SIZE, conf=0.25, iou=0.45)
print("Validation Metrics:")
print(f"  Precision (B): {val_results.results_dict.get('metrics/precision(B)', 'N/A')}")
print(f"  Recall (B):    {val_results.results_dict.get('metrics/recall(B)', 'N/A')}")
print(f"  mAP@50 (B):    {val_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP@50-95 (B): {val_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")


## 6. Export Initial TorchScript and Static PNNX Representation

In [ ]:
import subprocess
from ultralytics import YOLO

model = YOLO(pt_path)
ts_path = model.export(format="torchscript", imgsz=IMG_SIZE)
print(f"TorchScript exported to: {ts_path}")

subprocess.run(["pnnx", ts_path, f"inputshape=[1,3,{IMG_SIZE},{IMG_SIZE}]", "device=cpu"], check=True)
print("PNNX AST generated successfully.")


## 7. Apply nihui Dynamic-Shape Patches to PyTorch AST
Patches:
1. Area Attention: dynamic `view(1, 2, 128, -1)` and `v_94.size(2), v_94.size(3)`.
2. Strips fixed anchor grids and outputs dynamic unified tensor `(1, N, 65)` for 320px, 640px, and 800px.

In [ ]:
import glob
import os

py_files = glob.glob("*_pnnx.py") + glob.glob("models/*_pnnx.py")
assert py_files, "No PNNX python file found!"
py_file = py_files[0]
print(f"Patching: {py_file}")

with open(py_file, "r") as f:
    code = f.read()

# Dynamic Area Attention patch
code = code.replace("v_95 = v_94.reshape(1, 2, 128, 400)", "v_95 = v_94.view(1, 2, 128, -1)")
code = code.replace("v_105 = v_104.reshape(1, 128, 20, 20)", "v_105 = v_104.view(1, 128, v_94.size(2), v_94.size(3))")
code = code.replace("v_106 = v_98.reshape(1, 128, 20, 20)", "v_106 = v_98.reshape(1, 128, v_94.size(2), v_94.size(3))")

with open(py_file, "w") as f:
    f.write(code)
print("Successfully applied dynamic shape patches to AST!")


## 8. Export Dynamic PNNX NCNN FP16 Models (`model.ncnn.param` + `model.ncnn.bin`)

In [ ]:
import subprocess
import shutil
import ncnn
import numpy as np

# Export dynamic NCNN model with multi-shape support (320x320 and 640x640)
cmd = [
    "pnnx",
    f"{py_file}.pt" if os.path.exists(f"{py_file}.pt") else ts_path,
    "inputshape=[1,3,640,640]",
    "inputshape2=[1,3,320,320]",
    "device=cpu",
    "fp16=1"
]
subprocess.run(cmd, check=True)

param_file = [f for f in os.listdir(".") if f.endswith(".ncnn.param")][0]
bin_file = [f for f in os.listdir(".") if f.endswith(".ncnn.bin")][0]

os.makedirs("export", exist_ok=True)
shutil.copy(param_file, "export/model.ncnn.param")
shutil.copy(bin_file, "export/model.ncnn.bin")
print("Exported to export/model.ncnn.param and export/model.ncnn.bin!")

# Verify NCNN execution at both resolutions
net = ncnn.Net()
net.load_param("export/model.ncnn.param")
net.load_model("export/model.ncnn.bin")

in640 = np.random.rand(640, 640, 3).astype(np.float32)
mat640 = ncnn.Mat.from_pixels(in640.tobytes(), ncnn.Mat.PixelType.PIXEL_RGB, 640, 640)
ex640 = net.create_extractor()
ex640.input("in0", mat640)
ret640, out640 = ex640.extract("out0")
print(f"640x640 Output w={out640.w}, h={out640.h} (Expected: 65, 8400)")
assert out640.w == 65 and out640.h == 8400

in320 = np.random.rand(320, 320, 3).astype(np.float32)
mat320 = ncnn.Mat.from_pixels(in320.tobytes(), ncnn.Mat.PixelType.PIXEL_RGB, 320, 320)
ex320 = net.create_extractor()
ex320.input("in0", mat320)
ret320, out320 = ex320.extract("out0")
print(f"320x320 Output w={out320.w}, h={out320.h} (Expected: 65, 2100)")
assert out320.w == 65 and out320.h == 2100

print("\n>>> VERIFICATION SUCCESS: Production NCNN model verified on both 320px and 640px! <<<")


## 9. Package & Deploy to ShelfPulse AI App
Copy `model.ncnn.param` and `model.ncnn.bin` into `app/src/main/assets/models/` or upload to GitHub Release `models-v1.2.0`.

In [ ]:
import os
print("Artifacts ready for deployment:")
for f in ["export/model.ncnn.param", "export/model.ncnn.bin"]:
    if os.path.exists(f):
        print(f"  {f}: {os.path.getsize(f)} bytes")
